# Глубинное обучение для текстовых данных, ФКН ВШЭ
## Домашнее задание 3: Уменьшение размеров модели
### Оценивание и штрафы

Максимально допустимая оценка за работу — __16 баллов__.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

__Мягкий дедлайн 22.11.26 23:59__ \
__Жесткий дедлайн 24.11.26 23:59__

### О задании

В этом задании вам предстоит научиться решать задачу Named Entity Recognition (NER) на самом популярном датасете – [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). В вашем распоряжении будет предобученный BERT, который вам необходимо уменьшить с минимальными потерями в качестве до размера 20М параметров. Для этого вы самостоятельно реализуете факторизацию эмбеддингов, дистилляцию, шеринг параметров и так далее.

В этом задании вам придется проводить довольно много экспериментов, поэтому мы рекомендуем не писать весь код в тетрадке, а завести разные файлы для отдельных логических блоков и скомпоновать все в виде проекта. Это позволит вашему ноутбуку не разрастаться и сильно облегчит задачу и вам, и проверяющим. Так же постарайтесь логгировать все ваши эксперименты на wandb или других платформах, чтобы ничего не потерялось.

В этом задании __запрещается использовать агентов для написания кода__, но разрешается для помощи в чтении статей и осознании методов.

### Оценивание
Оценка за это домашнее задание будет формироваться из оценки за __задания__ и за __отчет__, в котором от вас требуется написать о проделанной работе. За отчет можно получить до 2-х баллов, однако в случае отсутствия отчета баллы за соответствующие задания не будут ставиться. Задания делятся на две части: _номерные_ и _на выбор_. За _номерные_ можно получить в сумме 6 баллов, за задания _на выбор_ можно получить до 8. То есть за все дз можно получить 16 баллов (но не радуйтесь рано, это не так просто). Все, что вы наберете свыше 10, будет считаться бонусами.

### О датасете

Named Entity Recognition – это задача классификации токенов по классам сущностей. В CoNLL-2003 для именования сущностей используется маркировка **BIO** (Beggining, Inside, Outside), в которой метки означают следующее:

- *B-{метка}* – начало сущности *{метка}*
- *I-{метка}* – продолжнение сущности *{метка}*
- *O* – не сущность

Существуют так же и другие способы маркировки, например, BILUO. Почитать о них можно [тут](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) и [тут](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

Всего в датасете есть 9 разных меток.
- O – слову не соответствует ни одна сущность.
- B-PER/I-PER – слово или набор слов соответстует определенному _человеку_.
- B-ORG/I-ORG – слово или набор слов соответстует определенной _организации_.
- B-LOC/I-LOC – слово или набор слов соответстует определенной _локации_.
- B-MISC/I-MISC – слово или набор слов соответстует сущности, которая не относится ни к одной из предыдущих. Например, национальность, произведение искусства, мероприятие и т.д.

Приступим!

In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/kaggle/working/ner-project")
REPO_URL = "https://github.com/roshlight/bert-ner-distillation"

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

In [1]:
%pip install -q -e /kaggle/working/ner-project

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ner-distill (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

PROJECT_DIR = "/kaggle/working/ner-project"
SRC_DIR = f"{PROJECT_DIR}/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import ner_distill

print("Пакет:", ner_distill.__file__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Пакет: /kaggle/working/ner-project/src/ner_distill/__init__.py
CUDA: True
GPU: Tesla T4


Начнем с загрузки и предобработки датасета.

In [4]:
from datasets import load_dataset

dataset = load_dataset("ashaba1in/conll2003")

dataset = dataset.remove_columns(["id", "pos_tags", "chunk_tags"])
dataset

README.md:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.22MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  309kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

In [5]:
from datasets import load_dataset

dataset2 = load_dataset("hgissbkh/conll2003-en")
dataset2

README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  838kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  212kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  192kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3252 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3454 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'ner'],
        num_rows: 14042
    })
    validation: Dataset({
        features: ['words', 'ner'],
        num_rows: 3252
    })
    test: Dataset({
        features: ['words', 'ner'],
        num_rows: 3454
    })
})

In [6]:
dataset['train'][1]

{'tokens': ['Peter', 'Blackburn'], 'ner_tags': [1, 2]}

In [7]:
dataset['train'][0]

{'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [8]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [9]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


### Предобработка

На протяжении всего домашнего задания мы будем использовать _cased_ версию BERT, то есть токенизатор будет учитывать регистр слов. Для задачи NER регистр важен, так как имена и названия организаций или предметов искусства часто пишутся с большой буквы, и будет глупо прятать от модели такую информацию.

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

При токенизации слова могут разделиться на несколько токенов (как слово `Fischler` из примера ниже), из-за чего появится несоответствие между числом токенов и меток. Это несоответствие нам придется устранить вручную.

In [11]:
example = dataset["train"][12]
words = example["tokens"]
tags = [label_names[t] for t in example["ner_tags"]]
tokenized_text = tokenizer(example["tokens"], is_split_into_words=True)

print('Слова: ', words)
print('Токены:', tokenized_text.tokens())
print('Метки:', tags)

Слова:  ['Only', 'France', 'and', 'Britain', 'backed', 'Fischler', "'s", 'proposal', '.']
Токены: ['[CLS]', 'Only', 'France', 'and', 'Britain', 'backed', 'Fi', '##sch', '##ler', "'", 's', 'proposal', '.', '[SEP]']
Метки: ['O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'O', 'O']


__Задание 1 (1 балл).__ Токенизируйте весь датасет и для каждого текста выравните токены с метками так, чтобы каждому токену соответствовала одна метка. При этом важно сохранить нотацию BIO. И не забудьте про специальные токены! Должно получиться что-то такое:

In [13]:
!git -C /kaggle/working/ner-project pull --ff-only

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 1), reused 5 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.17 KiB | 1.17 MiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   f8c9c24..d00788e  main       -> origin/main
Updating f8c9c24..d00788e
Fast-forward
 src/ner_distill/data.py | 75 +++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 75 insertions(+)
 create mode 100644 src/ner_distill/data.py


In [14]:
from ner_distill.data import (
    LABEL_NAMES,
    align_labels_with_tokens,
    load_ner_data,
)

dataset, tokenized_dataset, tokenizer, data_collator = load_ner_data()
label_names = LABEL_NAMES

tokenized_dataset

Токенизация и выравнивание BIO:   0%|          | 0/14041 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3250 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [15]:
aligned_labels = align_labels_with_tokens(example["ner_tags"], tokenized_text)
tags = [label_names[t] if t > -1 else t for t in aligned_labels]
print("Выровненные метки:", aligned_labels)
print("Выровненные названия меток:", tags)

Выровненные метки: [-100, 0, 5, 0, 5, 0, 1, 2, 2, 0, 0, 0, 0, -100]
Выровненные названия меток: [-100, 'O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', -100]


In [ ]:
# your code here

### Метрика

Для оценки качества NER обычно используют F1 меру с микро-усреднением. Мы загрузим ее из библиотеки `seqeval`. Функция `f1_score` принимает два 2d списка с правильными и предсказанными метками, записаными текстом, и возвращает для них значение F1. Вы можете использовать ее с параметрами по умолчанию.

In [16]:
! pip install seqeval

In [18]:
from seqeval.metrics import f1_score

Особенность подсчета F1 для NER заключается в том, что в некоторых ситуациях неправильные ответы могут засчитываться как правильные. Например, если модель предсказала `['I-PER', 'I-PER']`, то мы можем догадаться, что на самом деле должно быть `['B-PER', 'I-PER']`, так как сущность не может начинаться с `I-`. Функция `f1_score` учитывает это и поэтому работает только с текстовыми представлениями меток.

### Модель

В качестве базовой модели мы возьмем `bert-base-cased`. Как вы понимаете, он не обучался на задачу NER. Поэтому прежде чем приступать к уменьшению размера BERT, его необходимо дообучить.

__Задание 2 (1 балл)__ Дообучите `bert-base-cased` на нашем датасете с помощью обычного fine-tuning. У вас должно получиться хотя бы 0.9 F1 на тестовой выборке. Заметьте, что чем выше качество большой модели, тем лучше будет работать дистиллированный ученик. Для обучения можно использовать `Trainer` из Hugging Face.

In [19]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(label_names))

print('Число параметров:', sum(p.numel() for p in model.parameters()))

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Число параметров: 107726601


In [28]:
!git -C /kaggle/working/ner-project pull --ff-only

import importlib
import ner_distill.training

importlib.reload(ner_distill.training)

from ner_distill.training import create_teacher_trainer

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 415 bytes | 415.00 KiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   ea92a6c..8f1bbbe  main       -> origin/main
Updating ea92a6c..8f1bbbe
Fast-forward
 src/ner_distill/metrics.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [31]:
import importlib
import ner_distill.metrics as metrics_module
import ner_distill.training as training_module

importlib.reload(metrics_module)
importlib.reload(training_module)

from ner_distill.training import create_teacher_trainer
from ner_distill.metrics import make_compute_metrics

In [32]:
trainer = create_teacher_trainer(
    model=model,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.198707,0.146234,0.914443
2,0.112081,0.119167,0.938003
3,0.061311,0.109464,0.944291


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [34]:
teacher_path = "/kaggle/working/outputs/teacher/best"

trainer.save_model(teacher_path)
tokenizer.save_pretrained(teacher_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/outputs/teacher/best/tokenizer_config.json',
 '/kaggle/working/outputs/teacher/best/tokenizer.json')

### Факторизация матрицы эмбеддингов

Можно заметить, что на данный момент матрица эмбеддингов занимает $V \cdot H = 28996 \cdot 768 = 22.268.928$ параметров. Это aж пятая часть от всей модели! Давайте попробуем что-то с этим сделать. В модели [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) предлагается факторизовать матрицу эмбеддингов в произведение двух небольших матриц. Таким образом, параметры эмбеддингов будут содержать $V \cdot E + E \cdot H$ элементов, что гораздо меньше $V \cdot H$, если $H \gg E$. Авторы выбирают $E = 128$, однако ничего не мешает нам взять любое другое значение. Например, выбрав $H = 64$, мы уменьшим число параметров примерно на 20М.

__Задание 3 (1 балл).__ Напишите класс-обертку над слоем эмбеддингов, который реализует факторизацию на две матрицы, и дообучите факторизованную модель. Заметьте, обе матрицы можно инициализировать с помощью SVD разложения (а лучше [FWSVD](https://arxiv.org/pdf/2207.00112)!), чтобы начальное приближение было хорошим. Это сэкономит очень много времени на дообучении. С рангом разложения $H = 64$ у вас должно получиться F1 больше 0.87.

In [35]:
# your code here
!git -C /kaggle/working/ner-project pull --ff-only

import importlib
import ner_distill.factorization as factorization_module

importlib.reload(factorization_module)

from transformers import AutoModelForTokenClassification
from ner_distill.factorization import factorize_word_embeddings

factorized_model = AutoModelForTokenClassification.from_pretrained(
    teacher_path
)

params_before = sum(p.numel() for p in factorized_model.parameters())

factorized_model = factorize_word_embeddings(
    factorized_model,
    rank=64,
)

params_after = sum(p.numel() for p in factorized_model.parameters())

print("Параметров до:", params_before)
print("Параметров после:", params_after)
print("Убрано параметров:", params_before - params_after)

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.02 KiB | 1.02 MiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   8f1bbbe..f63e20e  main       -> origin/main
Updating 8f1bbbe..f63e20e
Fast-forward
 src/ner_distill/factorization.py | 74 ++++++++++++++++++++++++++++++++++++++++
 1 file changed, 74 insertions(+)
 create mode 100644 src/ner_distill/factorization.py


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Параметров до: 107726601
Параметров после: 87362569
Убрано параметров: 20364032


In [36]:
import gc
import torch

model.to("cpu")
del trainer

gc.collect()
torch.cuda.empty_cache()

In [37]:
from ner_distill.training import create_teacher_trainer

factorized_trainer = create_teacher_trainer(
    model=factorized_model,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir="/kaggle/working/outputs/factorized",
)

before_metrics = factorized_trainer.evaluate()
print("Validation F1 сразу после SVD:", before_metrics["eval_f1"])

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,F1
No log,3.167108,0,0.035847


Validation F1 сразу после SVD: 0.03584672435105068


In [38]:
factorized_train_result = factorized_trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.225414,0.207372,0.890692
2,0.151746,0.199976,0.917346
3,0.101885,0.179725,0.921324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [39]:
from ner_distill.metrics import make_compute_metrics

factorized_trainer.compute_metrics = make_compute_metrics(
    dataset["test"],
    tokenizer,
)

factorized_test_metrics = factorized_trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

print("Test F1:", factorized_test_metrics["test_f1"])

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,F1
0.101885,0.409149,3,0.870458


Test F1: 0.8704581358609794


In [40]:
from pathlib import Path

save_dir = Path("/kaggle/working/outputs/factorized/best")
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "rank": 64,
        "config": factorized_trainer.model.config.to_dict(),
        "state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in factorized_trainer.model.state_dict().items()
        },
    },
    save_dir / "factorized_model.pt",
)

tokenizer.save_pretrained(save_dir)
factorized_trainer.save_metrics("test", factorized_test_metrics)

### Дистилляция знаний

Дистилляция знаний – это парадигма обучения, в которой знания модели-учителя дистиллируются в модель-ученика. Учеником может быть произвольная модель меньшего размера, решающая ту же задачу, однако обычно ученик имеет ту же архитектуру, что и учитель. При дистилляции используются два функционала ошибки:

1. Стандартная кросс-энтропия.
1. Функция, задающая расстояние между распределениями предсказаний учителя и ученика. Чаще всего используют KL-дивергенцию.

Для того, чтобы распределение предсказаний учителя не было вырожденным, к softmax добавляют температуру больше 1, например, 2 или 5.   
__Важно:__ при делении логитов на температуру значения градиентов уменьшаются в $\tau^2$ раз (проверьте это!). Поэтому для возвращения их в изначальный масштаб ошибку надо домножить на $\tau^2$. Подробнее об этом можно почитать в разделе 2.1 [оригинальной статьи](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Задание 4 (3 балла).__ Реализуйте метод дистилляции знаний, изображенный на картинке. Для подсчета ошибки между предсказаниями ученика и учителя используйте KL-дивергенцию [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (обратите внимание на вормат ее входов). Для получения итоговой ошибки суммируйте мягкую ошибку с жесткой.   
В качестве учителя используйте дообученный BERT из задания 2. В качестве ученика возьмите необученную модель с размером __не больше 20M__ параметров. Вы можете использовать факторизацию матрицы эмбеддингов для уменьшения числа параметров. Если вы все сделали правильно, то на тестовой выборке вы должны получить значение F1 не меньше 0.7. Вам должно хватить примерно 20к итераций обучения для этого. Если у вас что-то не получается, то можно ориентироваться на статью про [DistilBERT](https://arxiv.org/abs/1910.01108) и на [эту статью](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).

__Важно:__
* Не забывайте добавлять _warmup_ при обучении ученика.
* Не забывайте переводить учителя в режим _eval_.

In [13]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/kaggle/working/ner-project")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/roshlight/bert-ner-distillation.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True,
    )

Already up to date.


In [14]:
%pip install -q -e /kaggle/working/ner-project

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ner-distill (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys
import importlib
import gc
import torch

SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

importlib.invalidate_caches()

import ner_distill.data as data_module
import ner_distill.metrics as metrics_module
import ner_distill.training as training_module
import ner_distill.models as models_module
import ner_distill.distillation as distillation_module

for module in (
    data_module,
    metrics_module,
    training_module,
    models_module,
    distillation_module,
):
    importlib.reload(module)

from transformers import AutoModelForTokenClassification, set_seed
from ner_distill.data import LABEL_NAMES, load_ner_data
from ner_distill.metrics import make_compute_metrics
from ner_distill.training import create_teacher_trainer
from ner_distill.models import create_student
from ner_distill.distillation import create_distillation_trainer

set_seed(42)

In [16]:
dataset, tokenized_dataset, tokenizer, data_collator = load_ner_data(
    max_length=256,
)

label_names = LABEL_NAMES

print(tokenized_dataset)

README.md:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.22MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  309kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

... (8 outputs left out)

Токенизация и выравнивание BIO:   0%|          | 0/14041 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3250 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [17]:
set_seed(42)

teacher_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_names),
)

teacher_trainer = create_teacher_trainer(
    model=teacher_model,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir="/kaggle/working/outputs/distillation_teacher",
)

teacher_train_result = teacher_trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Epoch,Training Loss,Validation Loss,F1
1,0.193178,0.175323,0.902000
2,0.115632,0.121132,0.940092
3,0.062019,0.114642,0.942891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [18]:
teacher_path = "/kaggle/working/outputs/distillation_teacher/best"

teacher_trainer.save_model(teacher_path)
tokenizer.save_pretrained(teacher_path)
teacher_trainer.save_metrics("train", teacher_train_result.metrics)

teacher_trainer.compute_metrics = make_compute_metrics(
    dataset["test"],
    tokenizer,
)

teacher_test_metrics = teacher_trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

teacher_trainer.save_metrics("test", teacher_test_metrics)

print("Teacher test F1:", teacher_test_metrics["test_f1"])
print("Путь до учителя:", teacher_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,F1
0.062019,0.317678,3,0.907065


Teacher test F1: 0.9070645019745502
Путь до учителя: /kaggle/working/outputs/distillation_teacher/best


In [19]:
del teacher_trainer, teacher_model
gc.collect()
torch.cuda.empty_cache()

set_seed(42)

teacher = AutoModelForTokenClassification.from_pretrained(
    teacher_path,
    local_files_only=True,
)

student = create_student(tokenizer)

print(
    "Параметров ученика:",
    sum(p.numel() for p in student.parameters()),
)

distillation_trainer = create_distillation_trainer(
    student=student,
    teacher=teacher,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    max_steps=20_000,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Параметров ученика: 10716425


In [20]:
distillation_result = distillation_trainer.train()

student_path = "/kaggle/working/outputs/student/best"

distillation_trainer.save_model(student_path)
tokenizer.save_pretrained(student_path)
distillation_trainer.save_metrics(
    "train",
    distillation_result.metrics,
)

print("Ученик сохранён:", student_path)

Step,Training Loss,Validation Loss,F1
1000,1.127211,1.265379,0.592500
2000,0.463503,0.851874,0.739191
3000,0.245326,0.690589,0.796342
4000,0.147317,0.739289,0.794380
5000,0.097538,0.665444,0.813794
6000,0.085608,0.785577,0.810495
7000,0.073040,0.685758,0.817624
8000,0.061734,0.681067,0.823372
8077,0.061734,1.013462,0.750825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [21]:
# your code here
distillation_trainer.compute_metrics = make_compute_metrics(
    dataset["test"],
    tokenizer,
)

student_test_metrics = distillation_trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

distillation_trainer.save_metrics("test", student_test_metrics)

print(f"Teacher test F1: {teacher_test_metrics['test_f1']:.4f}")
print(f"Student test F1: {student_test_metrics['test_f1']:.4f}")

Teacher test F1: 0.9071
Student test F1: 0.7508


# Задания на выбор

Как вы понимаете, есть еще довольно много разных способов уменьшить обученную модель. В этой секции вам предлагается реализовать разные техники на выбор. За каждую из них можно получить разное количество балов в зависимости от сложности. Успешность реализации будет оцениваться как по коду, так и по качеству на тестовой выборке. Все баллы за это дз, которые вы наберете сверх 10, будут считаться бонусными.   
В задании 4 вы обучали модель с ограничением числа параметров в 20М. При реализации техник из этой секции придерживайтесь такого же ограничения. Это позволит честно сравнивать методы между собой и делать правильные выводы. Напишите в отчете обо всем, что вы попробовали.

* __Шеринг весов (1 балл).__ В модификации BERT [ALBERT](https://arxiv.org/pdf/1909.11942) помимо факторизации эмбеддингов предлагается шерить веса между слоями. То есть разные слои используют одни и те же веса. Такая техника эвивалентна применению одного и того же слоя несколько раз. Она позволяет в несколько раз уменьшить число параметров и не сильно потерять в качестве.
* __Шеринг весов++ (0.5 балл).__ Вместо того, чтобы просто применять одни и те же слои, можно дообучить LoRA для каждого слоя, чтобы слои могли специализироваться. Ровно так делают в [этой статье](https://arxiv.org/pdf/2410.20672) от DeepMind.
* __Приближение промежуточных слоев (1.5 балла).__ Мы обсуждали, что помимо приближения выходов модели ученика к выходам модели учителя, можно приближать выходы промежуточных слоев. В [этой работе](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) подробно написано, как это можно сделать. Еще __+1 балл__ за реализацию метода из [MiniLMv2](https://arxiv.org/pdf/2012.15828).
* __Прунинг (3 балла).__ В методе [SparseGPT](https://arxiv.org/abs/2301.00774) предлагается подход, удаляющий веса модели один раз после обучения. При этом оказывается возможным удалить до половины всех весов без потери в качестве. Математика, стоящаяя за техникой, довольно сложная, однако общий подход простой – будем удалять веса в каждом слое по отдельности, при удалении части весов слоя, остальные веса будут перенастраиваться так, чтобы общий выход слоя не изменился. В качестве бейзлайна реализуйте более простой метод [Wanda](https://arxiv.org/pdf/2306.11695).
* __Инициализация ученика прунингом (2 балла).__ Обучение ученика с нуля занимает достаточно много времени. Вместо этого, ученика можно инициализоровать с помощью прунинга учителя, а затем дообучить с помощью дистилляции. Впервые так сделали в [Minitron](https://arxiv.org/pdf/2408.11796), а после в Ministral 3, Llama 3.2, Falcon3 и т.д. Важная проблема этого подхода заключается в том, что слои могут пруниться неравномерно, из-за чего при использовании модели будет сложнее распределять нагрузку. Но если задача – просто уменьшить размер, то это отличный способ.



# Sharing

In [1]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/kaggle/working/ner-project")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/roshlight/bert-ner-distillation.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True,
    )

Cloning into '/kaggle/working/ner-project'...


In [2]:
%pip install -q -e /kaggle/working/ner-project

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for ner-distill (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import gc
import json
import time
import importlib
import torch

SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

importlib.invalidate_caches()

for module_name in (
    "ner_distill.data",
    "ner_distill.metrics",
    "ner_distill.models",
    "ner_distill.training",
    "ner_distill.distillation",
    "ner_distill.sharing",
):
    module = importlib.import_module(module_name)
    importlib.reload(module)

from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

from ner_distill.data import LABEL_NAMES, load_ner_data
from ner_distill.metrics import make_compute_metrics
from ner_distill.models import create_student
from ner_distill.training import create_teacher_trainer
from ner_distill.distillation import DistillationTrainer
from ner_distill.sharing import (
    create_shared_student,
    load_shared_student,
    SaveBestSharedStudent,
)

assert torch.cuda.is_available(), "Включи GPU в настройках Kaggle"

OUTPUT_DIR = Path("/kaggle/working/outputs/sharing_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_PATH = OUTPUT_DIR / "teacher" / "best"
SHARED_PATH = OUTPUT_DIR / "student" / "best_shared.pt"

set_seed(42)

In [4]:
dataset, tokenized_dataset, tokenizer, data_collator = load_ner_data(
    max_length=256,
)

label_names = LABEL_NAMES

print(tokenized_dataset)

README.md:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.22MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  309kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

... (8 outputs left out)

Токенизация и выравнивание BIO:   0%|          | 0/14041 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3250 [00:00<?, ? examples/s]

Токенизация и выравнивание BIO:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [5]:
set_seed(42)

teacher_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_names),
)

teacher_trainer = create_teacher_trainer(
    model=teacher_model,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=str(OUTPUT_DIR / "teacher"),
)

teacher_started = time.perf_counter()
teacher_train_result = teacher_trainer.train()
teacher_seconds = time.perf_counter() - teacher_started

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Epoch,Training Loss,Validation Loss,F1
1,0.193178,0.175323,0.902000
2,0.115632,0.121132,0.940092
3,0.062019,0.114642,0.942891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [6]:
teacher_trainer.save_model(str(TEACHER_PATH))
tokenizer.save_pretrained(str(TEACHER_PATH))
teacher_trainer.save_metrics("train", teacher_train_result.metrics)

teacher_trainer.compute_metrics = make_compute_metrics(
    dataset["test"],
    tokenizer,
)

teacher_test_metrics = teacher_trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

teacher_trainer.save_metrics("test", teacher_test_metrics)

print("Teacher test F1:", teacher_test_metrics["test_f1"])
print("Время обучения, минут:", round(teacher_seconds / 60, 1))
print("Учитель сохранён:", TEACHER_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,F1
0.062019,0.317678,3,0.907065


Teacher test F1: 0.9070645019745502
Время обучения, минут: 6.1
Учитель сохранён: /kaggle/working/outputs/sharing_experiment/teacher/best


In [9]:
import gc
import torch
from pathlib import Path

for name in ("teacher_trainer", "teacher_model"):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

TEACHER_PATH = Path(
    "/kaggle/working/outputs/sharing_experiment/teacher/best"
)

teacher = AutoModelForTokenClassification.from_pretrained(
    str(TEACHER_PATH),
    local_files_only=True,
)

set_seed(42)
shared_student = create_shared_student(tokenizer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
best_shared = SaveBestSharedStudent(SHARED_PATH)

shared_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "student"),
    max_steps=5000,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="linear",
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none",
    seed=42,
)

shared_trainer = DistillationTrainer(
    model=shared_student,
    teacher=teacher,
    temperature=2.0,
    args=shared_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["validation"],
        tokenizer,
    ),
    callbacks=[best_shared],
)

In [11]:
shared_started = time.perf_counter()
shared_train_result = shared_trainer.train()

Step,Training Loss,Validation Loss,F1
500,1.403561,1.398829,0.587081
1000,0.584734,0.878785,0.717181
1500,0.351363,0.769889,0.760010
2000,0.205409,0.686980,0.787838
2500,0.153437,0.696092,0.793595



Сохранён лучший shared student: step=500, F1=0.5871

Сохранён лучший shared student: step=1000, F1=0.7172

Сохранён лучший shared student: step=1500, F1=0.7600

Сохранён лучший shared student: step=2000, F1=0.7878

Сохранён лучший shared student: step=2500, F1=0.7936


KeyboardInterrupt: 

In [13]:
from ner_distill.models import create_student

ordinary_model = create_student(tokenizer)
ordinary_count = sum(p.numel() for p in ordinary_model.parameters())
del ordinary_model

shared_count = sum(
    p.numel() for p in shared_trainer.model.parameters()
)

shared_seconds = time.perf_counter() - shared_started

run_summary = {
    "ordinary_student_parameters": ordinary_count,
    "shared_student_parameters": shared_count,
    "teacher_test_f1": float(teacher_test_metrics["test_f1"]),
    "best_validation_f1": best_shared.best_f1,
    "best_step": best_shared.best_step,
    "completed_steps": shared_trainer.state.global_step,
    "teacher_training_seconds": teacher_seconds,
    "shared_training_seconds": shared_seconds,
    "temperature": 2.0,
    "seed": 42,
}

del shared_trainer, shared_student, teacher
gc.collect()
torch.cuda.empty_cache()

best_model = load_shared_student(SHARED_PATH)

assert best_model.bert.encoder.layer[0] is best_model.bert.encoder.layer[2]
assert best_model.bert.encoder.layer[1] is best_model.bert.encoder.layer[3]

print("Лучший validation F1:", run_summary["best_validation_f1"])
print("Шаг лучшего checkpoint:", run_summary["best_step"])

Лучший validation F1: 0.7935949027498322
Шаг лучшего checkpoint: 2500


In [14]:
shared_evaluator = Trainer(
    model=best_model,
    args=TrainingArguments(
        output_dir=str(OUTPUT_DIR / "evaluation"),
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["test"],
        tokenizer,
    ),
)

shared_test_metrics = shared_evaluator.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

shared_evaluator.save_metrics("test", shared_test_metrics)
tokenizer.save_pretrained(str(SHARED_PATH.parent))

run_summary["shared_test_f1"] = float(shared_test_metrics["test_f1"])

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as file:
    json.dump(run_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(run_summary, ensure_ascii=False, indent=2))
print("\nРезультаты сохранены:", OUTPUT_DIR)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,F1
No log,0.856918,0,0.723738


{
  "ordinary_student_parameters": 10716425,
  "shared_student_parameters": 9136905,
  "teacher_test_f1": 0.9070645019745502,
  "best_validation_f1": 0.7935949027498322,
  "best_step": 2500,
  "completed_steps": 2599,
  "teacher_training_seconds": 365.3601413,
  "shared_training_seconds": 489.91686840700004,
  "temperature": 2.0,
  "seed": 42,
  "shared_test_f1": 0.7237381844123417
}

Результаты сохранены: /kaggle/working/outputs/sharing_experiment


# Sharing LoRA

In [15]:
!git -C /kaggle/working/ner-project pull --ff-only

import gc
import json
import importlib
from pathlib import Path

import torch
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

import ner_distill.sharing_lora as sharing_lora_module
importlib.reload(sharing_lora_module)

from ner_distill.sharing import (
    load_shared_student,
    SaveBestSharedStudent,
)
from ner_distill.sharing_lora import (
    add_layer_specific_lora,
    load_lora_shared_student,
)
from ner_distill.distillation import DistillationTrainer
from ner_distill.metrics import make_compute_metrics

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.24 KiB | 1.24 MiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   7a77c8f..a03833e  main       -> origin/main
Updating 7a77c8f..a03833e
Fast-forward
 src/ner_distill/sharing_lora.py | 73 +++++++++++++++++++++++++++++++++++++++++
 1 file changed, 73 insertions(+)
 create mode 100644 src/ner_distill/sharing_lora.py


In [16]:
BASE_DIR = Path("/kaggle/working/outputs/sharing_experiment")
LORA_DIR = Path("/kaggle/working/outputs/sharing_lora")
LORA_DIR.mkdir(parents=True, exist_ok=True)

for name in (
    "shared_trainer",
    "shared_evaluator",
    "shared_student",
    "best_model",
    "teacher",
    "lora_trainer",
    "lora_evaluator",
    "lora_student",
    "best_lora_model",
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

set_seed(42)

lora_student = load_shared_student(
    BASE_DIR / "student" / "best_shared.pt"
)

lora_student = add_layer_specific_lora(
    lora_student,
    rank=8,
    alpha=16,
)

teacher = AutoModelForTokenClassification.from_pretrained(
    str(BASE_DIR / "teacher" / "best"),
    local_files_only=True,
)

total_params = sum(p.numel() for p in lora_student.parameters())
trainable_params = sum(
    p.numel() for p in lora_student.parameters()
    if p.requires_grad
)

print("Всего параметров:", total_params)
print("Обучаемых параметров:", trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Всего параметров: 9169673
Обучаемых параметров: 32768


In [17]:
best_lora = SaveBestSharedStudent(
    LORA_DIR / "best_lora.pt"
)

lora_args = TrainingArguments(
    output_dir=str(LORA_DIR),
    max_steps=2000,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    weight_decay=0.0,
    warmup_steps=200,
    lr_scheduler_type="linear",
    fp16=True,
    eval_strategy="steps",
    eval_steps=250,
    logging_steps=50,
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none",
    seed=42,
)

lora_trainer = DistillationTrainer(
    model=lora_student,
    teacher=teacher,
    temperature=2.0,
    args=lora_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["validation"],
        tokenizer,
    ),
    callbacks=[best_lora],
)

In [18]:
initial_lora_metrics = lora_trainer.evaluate()

print(
    "Validation F1 до обучения LoRA:",
    initial_lora_metrics["eval_f1"],
)


Сохранён лучший shared student: step=0, F1=0.7936


Training Loss,Validation Loss,Step,F1
No log,0.696092,0,0.793595


Validation F1 до обучения LoRA: 0.7935949027498322


In [19]:
lora_train_result = lora_trainer.train()

Step,Training Loss,Validation Loss,F1
250,0.142649,0.669127,0.801710
500,0.130428,0.664174,0.804134
750,0.117036,0.653425,0.802412
1000,0.119402,0.651357,0.803690
1250,0.103346,0.651577,0.804297
1500,0.107134,0.649613,0.803724
1750,0.109290,0.650047,0.804260
2000,0.101970,0.649984,0.804798



Сохранён лучший shared student: step=250, F1=0.8017

Сохранён лучший shared student: step=500, F1=0.8041

Сохранён лучший shared student: step=1250, F1=0.8043

Сохранён лучший shared student: step=2000, F1=0.8048


In [20]:
lora_summary = {
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "initial_validation_f1": float(initial_lora_metrics["eval_f1"]),
    "best_validation_f1": best_lora.best_f1,
    "best_step": best_lora.best_step,
    "completed_steps": lora_trainer.state.global_step,
    "rank": 8,
    "alpha": 16,
}

for name in ("lora_trainer", "lora_student", "teacher"):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

best_lora_model = load_lora_shared_student(
    LORA_DIR / "best_lora.pt"
)

print("Лучший validation F1:", lora_summary["best_validation_f1"])
print("Лучший шаг:", lora_summary["best_step"])

Лучший validation F1: 0.8047982551799346
Лучший шаг: 2000


In [21]:
lora_evaluator = Trainer(
    model=best_lora_model,
    args=TrainingArguments(
        output_dir=str(LORA_DIR / "evaluation"),
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["test"],
        tokenizer,
    ),
)

lora_test_metrics = lora_evaluator.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

lora_evaluator.save_metrics("test", lora_test_metrics)
tokenizer.save_pretrained(str(LORA_DIR))

lora_summary["test_f1"] = float(lora_test_metrics["test_f1"])

with open(LORA_DIR / "summary.json", "w", encoding="utf-8") as file:
    json.dump(lora_summary, file, ensure_ascii=False, indent=2)

print("Shared + LoRA test F1:", lora_test_metrics["test_f1"])
print("Результаты сохранены:", LORA_DIR)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,F1
No log,0.791690,0,0.739312


Shared + LoRA test F1: 0.73931206115012
Результаты сохранены: /kaggle/working/outputs/sharing_lora


# Hidden distillation

In [22]:
!git -C /kaggle/working/ner-project pull --ff-only

import gc
import json
import time
import importlib
from pathlib import Path

import torch
from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

import ner_distill.hidden_distillation as hidden_module
importlib.reload(hidden_module)

from ner_distill.hidden_distillation import (
    HiddenStudent,
    create_hidden_student,
    create_hidden_trainer,
)
from ner_distill.metrics import make_compute_metrics

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.88 KiB | 1.88 MiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   a03833e..384a110  main       -> origin/main
Updating a03833e..384a110
Fast-forward
 src/ner_distill/hidden_distillation.py | 172 +++++++++++++++++++++++++++++++++
 1 file changed, 172 insertions(+)
 create mode 100644 src/ner_distill/hidden_distillation.py


In [23]:
for name in (
    "shared_trainer",
    "shared_evaluator",
    "shared_student",
    "best_model",
    "lora_trainer",
    "lora_evaluator",
    "lora_student",
    "best_lora_model",
    "teacher",
    "hidden_trainer",
    "hidden_evaluator",
    "hidden_student",
    "best_hidden_model",
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

TEACHER_PATH = Path(
    "/kaggle/working/outputs/sharing_experiment/teacher/best"
)
HIDDEN_DIR = Path("/kaggle/working/outputs/hidden_distillation")
HIDDEN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(42)

teacher = AutoModelForTokenClassification.from_pretrained(
    str(TEACHER_PATH),
    local_files_only=True,
)

hidden_student = create_hidden_student(
    tokenizer,
    teacher_hidden_size=teacher.config.hidden_size,
)

hidden_parameter_count = sum(
    p.numel() for p in hidden_student.parameters()
)

print("Параметров ученика вместе с проекциями:", hidden_parameter_count)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Параметров ученика вместе с проекциями: 11502857


In [24]:
hidden_trainer = create_hidden_trainer(
    student=hidden_student,
    teacher=teacher,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=HIDDEN_DIR,
    max_steps=5000,
)

In [29]:
import gc
import sys
import torch

for name in ("hidden_trainer", "hidden_student", "teacher"):
    globals().pop(name, None)

for attribute in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, attribute):
        delattr(sys, attribute)

gc.collect()
torch.cuda.empty_cache()

In [30]:
from transformers import AutoModelForTokenClassification, set_seed

set_seed(42)

teacher = AutoModelForTokenClassification.from_pretrained(
    str(TEACHER_PATH),
    local_files_only=True,
)

hidden_student = create_hidden_student(
    tokenizer,
    teacher_hidden_size=teacher.config.hidden_size,
)

hidden_trainer = create_hidden_trainer(
    student=hidden_student,
    teacher=teacher,
    dataset=dataset,
    tokenized_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=HIDDEN_DIR,
    max_steps=5000,
)

hidden_trainer.args.per_device_train_batch_size = 4
hidden_trainer.args.per_device_eval_batch_size = 4
hidden_trainer.args.gradient_accumulation_steps = 4

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [31]:
import time
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(HIDDEN_DIR))
print("Продолжаем с:", last_checkpoint)

hidden_started = time.perf_counter()

hidden_train_result = hidden_trainer.train(
    resume_from_checkpoint=last_checkpoint,
)

Продолжаем с: None


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 20.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 13.45 GiB is allocated by PyTorch, and 899.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
hidden_summary = {
    "parameters_with_projections": hidden_parameter_count,
    "best_validation_f1": float(hidden_trainer.state.best_metric),
    "best_checkpoint": hidden_trainer.state.best_model_checkpoint,
    "completed_steps": hidden_trainer.state.global_step,
    "training_seconds": time.perf_counter() - hidden_started,
    "teacher_layers": [3, 6, 9, 12],
    "temperature": 2.0,
    "hidden_loss_weight": 1.0,
    "seed": 42,
}

best_checkpoint = hidden_trainer.state.best_model_checkpoint

for name in ("hidden_trainer", "hidden_student", "teacher"):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

best_hidden_model = HiddenStudent.from_pretrained(
    best_checkpoint,
    local_files_only=True,
)

best_hidden_model.save_pretrained(str(HIDDEN_DIR / "best"))
tokenizer.save_pretrained(str(HIDDEN_DIR / "best"))

print("Лучший validation F1:", hidden_summary["best_validation_f1"])
print("Лучший checkpoint:", best_checkpoint)

In [ ]:
hidden_evaluator = Trainer(
    model=best_hidden_model,
    args=TrainingArguments(
        output_dir=str(HIDDEN_DIR / "evaluation"),
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["test"],
        tokenizer,
    ),
)

hidden_test_metrics = hidden_evaluator.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

hidden_evaluator.save_metrics("test", hidden_test_metrics)

hidden_summary["test_f1"] = float(hidden_test_metrics["test_f1"])

with open(HIDDEN_DIR / "summary.json", "w", encoding="utf-8") as file:
    json.dump(hidden_summary, file, ensure_ascii=False, indent=2)

print("Hidden distillation test F1:", hidden_test_metrics["test_f1"])

# Pruning

In [37]:
import subprocess

subprocess.run(
    ["git", "-C", "/kaggle/working/ner-project", "pull", "--ff-only"],
    check=True,
)

Already up to date.


CompletedProcess(args=['git', '-C', '/kaggle/working/ner-project', 'pull', '--ff-only'], returncode=0)

In [38]:
import sys
import gc
import json
import time
import importlib
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import Trainer, TrainingArguments

SRC_DIR = "/kaggle/working/ner-project/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

importlib.invalidate_caches()

import ner_distill.data as data_module
import ner_distill.metrics as metrics_module
import ner_distill.sharing as sharing_module
import ner_distill.pruning as pruning_module

for module in (
    data_module,
    metrics_module,
    sharing_module,
    pruning_module,
):
    importlib.reload(module)

from ner_distill.data import load_ner_data
from ner_distill.metrics import make_compute_metrics
from ner_distill.sharing import (
    load_shared_student,
    save_shared_student,
)
from ner_distill.pruning import (
    prune_bert,
    sparsity_statistics,
)

In [39]:
for name in (
    "trainer",
    "teacher_trainer",
    "teacher_model",
    "teacher",
    "model",
    "evaluator",
    "factorized_trainer",
    "factorized_model",
    "distillation_trainer",
    "student_evaluator",
    "student",
    "best_student",
    "shared_trainer",
    "shared_evaluator",
    "shared_student",
    "best_model",
    "lora_trainer",
    "lora_evaluator",
    "lora_student",
    "best_lora_model",
    "hidden_trainer",
    "hidden_evaluator",
    "hidden_student",
    "best_hidden_model",
    "batch",
    "loss",
):
    globals().pop(name, None)

for attribute in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, attribute):
        delattr(sys, attribute)

gc.collect()
torch.cuda.empty_cache()

In [40]:
BASE_PATH = Path(
    "/kaggle/working/outputs/sharing_experiment/student/best_shared.pt"
)

PRUNING_DIR = Path("/kaggle/working/outputs/pruning_shared")
PRUNING_DIR.mkdir(parents=True, exist_ok=True)

dataset, tokenized_dataset, tokenizer, data_collator = load_ner_data(
    max_length=256,
)

In [41]:
calibration_dataset = (
    tokenized_dataset["train"]
    .shuffle(seed=42)
    .select(range(512))
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator,
)

validation_metrics = make_compute_metrics(
    dataset["validation"],
    tokenizer,
)

test_metrics = make_compute_metrics(
    dataset["test"],
    tokenizer,
)

In [42]:
pruning_results = []

for method in ("baseline", "wanda", "sparsegpt"):
    print(f"\n===== {method} =====")

    model = load_shared_student(BASE_PATH)
    model.to("cuda").float().eval()

    pruning_seconds = 0.0

    if method != "baseline":
        torch.cuda.synchronize()
        started = time.perf_counter()

        model = prune_bert(
            model=model,
            calibration_loader=calibration_loader,
            method=method,
            sparsity=0.5,
            block_size=128,
            damping=0.01,
            device="cuda",
        )

        torch.cuda.synchronize()
        pruning_seconds = time.perf_counter() - started

    statistics = sparsity_statistics(model)

    # Сохраняем результат прунинга до оценки качества.
    if method != "baseline":
        model_dir = PRUNING_DIR / method
        save_shared_student(model, model_dir / "model.pt")
        tokenizer.save_pretrained(str(model_dir))

    evaluator = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(PRUNING_DIR / method / "evaluation"),
            per_device_eval_batch_size=4,
            report_to="none",
        ),
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=validation_metrics,
    )

    validation_result = evaluator.evaluate(
        eval_dataset=tokenized_dataset["validation"],
        metric_key_prefix="validation",
    )

    evaluator.compute_metrics = test_metrics

    test_result = evaluator.evaluate(
        eval_dataset=tokenized_dataset["test"],
        metric_key_prefix="test",
    )

    result = {
        "method": method,
        "validation_f1": float(validation_result["validation_f1"]),
        "test_f1": float(test_result["test_f1"]),
        "pruning_seconds": pruning_seconds,
        **statistics,
    }

    pruning_results.append(result)

    with open(
        PRUNING_DIR / "results.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            pruning_results,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(result)

    del evaluator, model
    gc.collect()
    torch.cuda.empty_cache()


===== baseline =====


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,F1
No log,0.496681,0,0.793595


Training Loss,Validation Loss,Step,F1
No log,0.872163,0,0.723738


{'method': 'baseline', 'validation_f1': 0.7935949027498322, 'test_f1': 0.7237381844123417, 'pruning_seconds': 0.0, 'total_parameters': 9136905, 'nonzero_parameters': 9136649, 'whole_model_sparsity': 2.801824031217759e-05, 'encoder_linear_sparsity': 0.0}

===== wanda =====
wanda: обработан слой 1/4
wanda: обработан слой 2/4
wanda: обработан слой 3/4
wanda: обработан слой 4/4


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,F1
No log,0.501351,0,0.792006


Training Loss,Validation Loss,Step,F1
No log,0.873698,0,0.719358


{'method': 'sparsegpt', 'validation_f1': 0.7920060458476783, 'test_f1': 0.71935771632471, 'pruning_seconds': 11.805753558000106, 'total_parameters': 9136905, 'nonzero_parameters': 8350217, 'whole_model_sparsity': 0.0861000524794775, 'encoder_linear_sparsity': 0.5}


In [43]:
pruning_table = pd.DataFrame(pruning_results)

baseline_f1 = pruning_table.loc[
    pruning_table["method"] == "baseline",
    "test_f1",
].iloc[0]

pruning_table["test_f1_drop"] = (
    baseline_f1 - pruning_table["test_f1"]
)

pruning_table.to_csv(
    PRUNING_DIR / "results.csv",
    index=False,
)

display(pruning_table[[
    "method",
    "total_parameters",
    "nonzero_parameters",
    "encoder_linear_sparsity",
    "whole_model_sparsity",
    "validation_f1",
    "test_f1",
    "test_f1_drop",
    "pruning_seconds",
]])

,method,total_parameters,nonzero_parameters,encoder_linear_sparsity,whole_model_sparsity,validation_f1,test_f1,test_f1_drop,pruning_seconds
0,baseline,9136905,9136649,0.0,0.000028,0.793595,0.723738,0.000000,0.000000
1,wanda,9136905,8350217,0.5,0.086100,0.778999,0.707909,0.015829,10.183260
2,sparsegpt,9136905,8350217,0.5,0.086100,0.792006,0.719358,0.004380,11.805754


# Pruning init

In [45]:
!git -C /kaggle/working/ner-project pull --ff-only

import gc
import json
import time
import importlib
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

import ner_distill.structured_pruning as structured_module
importlib.reload(structured_module)

from ner_distill.structured_pruning import (
    PrunedStudent,
    select_ffn_neurons,
    build_pruned_student,
)
from ner_distill.data import load_ner_data
from ner_distill.metrics import make_compute_metrics
from ner_distill.distillation import DistillationTrainer

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.68 KiB | 1.68 MiB/s, done.
From https://github.com/roshlight/bert-ner-distillation
   894493b..bcaedc3  main       -> origin/main
Updating 894493b..bcaedc3
Fast-forward
 src/ner_distill/structured_pruning.py | 152 ++++++++++++++++++++++++++++++++++
 1 file changed, 152 insertions(+)


In [46]:
for name in (
    "model",
    "evaluator",
    "teacher",
    "hidden_trainer",
    "hidden_evaluator",
    "hidden_student",
    "best_hidden_model",
    "shared_trainer",
    "shared_evaluator",
    "shared_student",
    "best_model",
    "lora_trainer",
    "lora_evaluator",
    "lora_student",
    "best_lora_model",
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

TEACHER_PATH = Path(
    "/kaggle/working/outputs/sharing_experiment/teacher/best"
)
STRUCTURED_DIR = Path("/kaggle/working/outputs/structured_student")
STRUCTURED_DIR.mkdir(parents=True, exist_ok=True)

set_seed(42)

dataset, tokenized_dataset, tokenizer, data_collator = load_ner_data(
    max_length=256,
)

teacher = AutoModelForTokenClassification.from_pretrained(
    str(TEACHER_PATH),
    local_files_only=True,
)

calibration_dataset = (
    tokenized_dataset["train"]
    .shuffle(seed=42)
    .select(range(512))
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [47]:
selected_neurons = select_ffn_neurons(
    teacher=teacher,
    calibration_loader=calibration_loader,
    layer_indices=(2, 5, 8, 11),
    intermediate_size=1024,
)

pruned_student = build_pruned_student(
    teacher=teacher,
    selected_neurons=selected_neurons,
    layer_indices=(2, 5, 8, 11),
    intermediate_size=1024,
    embedding_rank=64,
)

gc.collect()
torch.cuda.empty_cache()

pruned_parameter_count = sum(
    p.numel() for p in pruned_student.parameters()
)

print("Параметров ученика:", pruned_parameter_count)

pruned_student.save_pretrained(
    str(STRUCTURED_DIR / "initial")
)
tokenizer.save_pretrained(
    str(STRUCTURED_DIR / "initial")
)

Параметров ученика: 18068489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/outputs/structured_student/initial/tokenizer_config.json',
 '/kaggle/working/outputs/structured_student/initial/tokenizer.json')

In [48]:
structured_args = TrainingArguments(
    output_dir=str(STRUCTURED_DIR / "training"),
    max_steps=3000,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    warmup_steps=300,
    lr_scheduler_type="linear",
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

structured_trainer = DistillationTrainer(
    model=pruned_student,
    teacher=teacher,
    temperature=2.0,
    args=structured_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["validation"],
        tokenizer,
    ),
)

In [49]:
structured_started = time.perf_counter()
structured_train_result = structured_trainer.train()

Step,Training Loss,Validation Loss,F1
500,0.724183,0.612454,0.789820
1000,0.422290,0.414073,0.846683
1500,0.322999,0.357819,0.868821
2000,0.235443,0.338319,0.878636
2500,0.207098,0.319444,0.882996
3000,0.187641,0.316410,0.883097


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [51]:
from transformers import Trainer, TrainingArguments

initial_model = PrunedStudent.from_pretrained(
    str(STRUCTURED_DIR / "initial"),
    local_files_only=True,
)

initial_evaluator = Trainer(
    model=initial_model,
    args=TrainingArguments(
        output_dir=str(STRUCTURED_DIR / "initial_evaluation"),
        use_cpu=True,
        per_device_eval_batch_size=4,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["validation"],
        tokenizer,
    ),
)

before_metrics = initial_evaluator.evaluate(
    eval_dataset=tokenized_dataset["validation"],
)

initial_evaluator.save_metrics("eval", before_metrics)

print("Validation F1 до дообучения:", before_metrics["eval_f1"])

del initial_evaluator, initial_model
gc.collect()

Loading weights:   0%|          | 0/72 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,F1
No log,1.554849,0,0.105769


Validation F1 до дообучения: 0.10576923076923078


314

In [54]:
import json
from pathlib import Path

from transformers.trainer_utils import get_last_checkpoint
from ner_distill.structured_pruning import PrunedStudent

STRUCTURED_DIR = Path("/kaggle/working/outputs/structured_student")

last_checkpoint = get_last_checkpoint(
    str(STRUCTURED_DIR / "training")
)

print("Последний checkpoint:", last_checkpoint)

with open(
    Path(last_checkpoint) / "trainer_state.json",
    encoding="utf-8",
) as file:
    trainer_state = json.load(file)

initial_f1 = float(before_metrics["eval_f1"])
trained_f1 = trainer_state.get("best_metric")

best_path = STRUCTURED_DIR / "initial"
selected_validation_f1 = initial_f1

if trained_f1 is not None and trained_f1 > initial_f1:
    # Восстанавливаем путь внутри текущей папки эксперимента.
    checkpoint_name = Path(
        trainer_state["best_model_checkpoint"]
    ).name
    best_path = STRUCTURED_DIR / "training" / checkpoint_name
    selected_validation_f1 = float(trained_f1)

best_pruned_model = PrunedStudent.from_pretrained(
    str(best_path),
    local_files_only=True,
)

structured_summary = {
    "parameters": sum(p.numel() for p in best_pruned_model.parameters()),
    "initial_validation_f1": initial_f1,
    "selected_validation_f1": selected_validation_f1,
    "selected_checkpoint": str(best_path),
    "last_saved_step": trainer_state["global_step"],
    "teacher_layers_one_based": [3, 6, 9, 12],
    "ffn_size": 1024,
    "embedding_rank": 64,
}

best_pruned_model.save_pretrained(str(STRUCTURED_DIR / "best"))
tokenizer.save_pretrained(str(STRUCTURED_DIR / "best"))

with open(
    STRUCTURED_DIR / "summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(structured_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(structured_summary, ensure_ascii=False, indent=2))

Последний checkpoint: /kaggle/working/outputs/structured_student/training/checkpoint-3000


Loading weights:   0%|          | 0/72 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "parameters": 18068489,
  "initial_validation_f1": 0.10576923076923078,
  "selected_validation_f1": 0.883097418817652,
  "selected_checkpoint": "/kaggle/working/outputs/structured_student/training/checkpoint-3000",
  "last_saved_step": 3000,
  "teacher_layers_one_based": [
    3,
    6,
    9,
    12
  ],
  "ffn_size": 1024,
  "embedding_rank": 64
}


In [55]:
structured_evaluator = Trainer(
    model=best_pruned_model,
    args=TrainingArguments(
        output_dir=str(STRUCTURED_DIR / "evaluation"),
        per_device_eval_batch_size=4,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(
        dataset["test"],
        tokenizer,
    ),
)

structured_test_metrics = structured_evaluator.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

structured_evaluator.save_metrics("test", structured_test_metrics)

structured_summary["test_f1"] = float(
    structured_test_metrics["test_f1"]
)

with open(
    STRUCTURED_DIR / "summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        structured_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(structured_summary, ensure_ascii=False, indent=2))

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,F1
No log,0.514648,0,0.848242


{
  "parameters": 18068489,
  "initial_validation_f1": 0.10576923076923078,
  "selected_validation_f1": 0.883097418817652,
  "selected_checkpoint": "/kaggle/working/outputs/structured_student/training/checkpoint-3000",
  "last_saved_step": 3000,
  "teacher_layers_one_based": [
    3,
    6,
    9,
    12
  ],
  "ffn_size": 1024,
  "embedding_rank": 64,
  "test_f1": 0.8482415568548738
}
